In [1]:
!pip install pytorchvideo torchvision av

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.7/132.7 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 33.7 MB/s eta 0:00:00
  Created wheel for pytorchvideo: filename=pytorchvideo-0.1.5-py3-none-any.whl size=188686 sha256=853925f6647596b76b8daeb4d39db4958a8ca028b703b2a0c702c9d9c62fda74
  Stored in directory: /root/.cache/pip/wheels/b3/49/dc/aab2dce83e38b59849db13a4f4ddd220e568e24b58332fb0f9
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61397 sha256=75eb2e1d6c3c269290a907f1bbedf4ecfa55ced7545921fe4638f525a77ef763
  Stored in directory: /root/.cache/pip/wheels/ed/9f/a5/e4f5b27454ccd4596bd8b62432c7d6b1ca9fa22aef9d70a16a
  Created wheel f

In [2]:
# --- 0. THE PYTORCHVIDEO BUG FIX (Monkey Patch) ---
import sys
import torchvision.transforms.functional as F_vision
sys.modules['torchvision.transforms.functional_tensor'] = F_vision

# --- 1. IMPORTS ---
import os
import shutil
import torch
import torch.nn as nn
import torch.nn.functional as F
from google.colab import drive

from torchvision.models.video import swin3d_t
from torchvision.transforms import Compose, Lambda, Resize
from torchvision.transforms._transforms_video import NormalizeVideo
from pytorchvideo.transforms import ApplyTransformToKey, UniformTemporalSubsample
from pytorchvideo.data.encoded_video import EncodedVideo

# --- 2. CONFIGURATION ---
# Connect to Google Drive
drive.mount('/content/drive', force_remount=True)

CLASSES = ['walking', 'running']
NUM_FRAMES = 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# The Paths
MODEL_WEIGHTS_PATH = "/content/drive/MyDrive/video_swin3d_TrainedModel.pth"
TEST_FOLDER_PATH = "/content/drive/MyDrive/video_data/test"
RESULT_FOLDER_PATH = "/content/drive/MyDrive/Result"

# Mapping the guesses to your EXACT existing folders
BUCKET_NAMES = {
    'walking': 'Predicted_Walking',
    'running': 'Predicted_Running'
}

# --- 3. BUILD THE ROBOT BODY & LOAD THE BRAIN ---
print("Building the robot and loading the brain...")
model = swin3d_t()
model.head = nn.Linear(model.head.in_features, len(CLASSES))
model.load_state_dict(torch.load(MODEL_WEIGHTS_PATH, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()

# --- 4. VIDEO GLASSES (TRANSFORMS) ---
video_transform = Compose([
    ApplyTransformToKey(
        key="video",
        transform=Compose([
            UniformTemporalSubsample(NUM_FRAMES),
            Lambda(lambda x: x / 255.0),
            NormalizeVideo(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            Resize((224, 224))
        ])
    )
])

# --- 5. THE SORTING LOOP ---
print("\n--- STARTING THE SORTING MACHINE ---")

# Look at every single file inside the test folder
for filename in os.listdir(TEST_FOLDER_PATH):

    # Only look at video files, ignore random hidden files
    if not filename.lower().endswith(('.mp4', '.avi', '.mov')):
        continue

    video_path = os.path.join(TEST_FOLDER_PATH, filename)
    print(f"\nWatching: {filename}...")

    try:
        # Load the 2-second clip and put the glasses on it
        video = EncodedVideo.from_path(video_path)
        video_data = video.get_clip(start_sec=0.0, end_sec=2.0)
        video_data = video_transform(video_data)
        input_tensor = video_data["video"].unsqueeze(0).to(DEVICE)

        # Make the guess!
        with torch.no_grad():
            output = model(input_tensor)
            probs = F.softmax(output, dim=1)
            conf, pred_idx = torch.max(probs, 1)

        # Figure out the winning guess
        winning_guess = CLASSES[pred_idx.item()]
        confidence_score = conf.item() * 100

        print(f"Guess: {winning_guess.upper()} ({confidence_score:.2f}% sure)")

        # --- THE FILE MOVING MAGIC ---
        # Look up the exact folder name from your dictionary
        target_bucket = BUCKET_NAMES[winning_guess]

        # Build the exact path to your existing folder
        target_path = os.path.join(RESULT_FOLDER_PATH, target_bucket, filename)

        # Copy the video into your existing folder!
        shutil.copy(video_path, target_path)
        print(f"-> Safely copied into {target_bucket}!")

    except Exception as e:
        print(f"Oh no! Something went wrong with {filename}: {e}")

print("\n--- ALL DONE! CHECK YOUR RESULT FOLDERS! ---")

/usr/local/lib/python3.12/dist-packages/torchvision/transforms/_functional_video.py:6: UserWarning: The 'torchvision.transforms._functional_video' module is deprecated since 0.12 and will be removed in the future. Please use the 'torchvision.transforms.functional' module instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/transforms/_transforms_video.py:22: UserWarning: The 'torchvision.transforms._transforms_video' module is deprecated since 0.12 and will be removed in the future. Please use the 'torchvision.transforms' module instead.
  warnings.warn(


Mounted at /content/drive
Building the robot and loading the brain...

--- STARTING THE SORTING MACHINE ---

Watching: v_PlayingCello_g04_c02.avi...
Guess: RUNNING (99.87% sure)
-> Safely copied into Predicted_Running!

Watching: v_TennisSwing_g05_c07.avi...
Guess: WALKING (100.00% sure)
-> Safely copied into Predicted_Walking!

Watching: v_ShavingBeard_g03_c01.avi...
Guess: RUNNING (96.23% sure)
-> Safely copied into Predicted_Running!

--- ALL DONE! CHECK YOUR RESULT FOLDERS! ---
